*Module 8 of 9*

> **¿Prefieres español?** Abre [`08_proyecto_mapa_de_cultivos.ipynb`](../es/08_proyecto_mapa_de_cultivos.ipynb) — es el mismo módulo, en español.


# 🌾 Module 8 — Capstone: the crop map, end to end

This is the moment everything comes together. You will run the **complete
pipeline in one sitting** — load the tile, look at it, segment it, turn
parcels into a table, attach the field labels, train the classifier and
paint the crop map.

Nothing here is new: every step below is one you already understand from
modules 2–7. This time, notice how *short* the whole story is when you
know what each line does.

🧭 **Objectives**
- Run the full crop-classification pipeline without help.
- Read the quality report and the final map critically.


In [ ]:
# Step 0 — Where is this Python running?
import sys, platform
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {sys.platform!r} / {platform.machine()!r}")
if sys.platform == "emscripten":
    print("Running in WebAssembly, INSIDE your browser. No server. 🚀")
else:
    print("Running locally (regular Python) — everything works the same.")

## Where does the data come from?

The tile you are about to load is a **real product**: the **March-2018
geomedian** of the Yaqui Valley, built by `geocrop_analysis_mx` from
**NASA HLS** imagery (Landsat + Sentinel-2 harmonized), streamed from open
**STAC/COG** catalogs. No Google Earth Engine was needed — though the
pipeline can optionally use GEE if you have an account.

![data sources](../../anim/en/02_data_sources.svg)

![the geomedian](../../anim/en/03_geomedian.svg)


In [ ]:
# Step 1 — Tools + workshop data (a few MB; cached after the first run)
%pip install -q shepherd-wasm
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"{RAW}/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

RAW = "https://raw.githubusercontent.com/abxda/portable-geocrop/main/files"
TILE   = await get_file("crop_tile_384.tif")
LABELS = await get_file("crop_labels_384.tif")
NAMES  = await get_file("class_names.json")
print("Ready:", TILE, LABELS, NAMES)

## Look at the field: true color and NDVI

13 layers per pixel: 6 spectral bands plus 7 vegetation/soil indices,
already computed in the geomedian.

![bands and NDVI](../../anim/en/04_bands_ndvi.svg)


In [ ]:
# Step 2 — RGB and NDVI
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()                      # (13, 384, 384) int16
    band_names = list(src.descriptions)
print("Bands:", band_names)

rgb  = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
ndvi = img[6] / 10000.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))
ax1.imshow(rgb); ax1.set_title("True color (geomedian, March 2018)"); ax1.axis("off")
im = ax2.imshow(ndvi, cmap="RdYlGn", vmin=0, vmax=0.9)
ax2.set_title("NDVI — crop vigor"); ax2.axis("off")
plt.colorbar(im, ax=ax2, shrink=0.8); plt.tight_layout(); plt.show()

## From pixels to parcels: Shepherd segmentation

![segmentation](../../anim/en/05_segmentation.svg)


In [ ]:
# Step 3 — Segment the tile into homogeneous parcels (~30-60 s)
import shepherd_wasm, time

t0 = time.time()
result = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0,
    fixedKMeansInit=True)
seg = result.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcels in {time.time()-t0:.1f} s")

from scipy import ndimage
edges = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = rgb.copy(); vis[edges] = [1, 1, 0]
plt.figure(figsize=(7.5, 7.5)); plt.imshow(vis)
plt.title(f"{n_seg} parcels (yellow = boundaries)"); plt.axis("off"); plt.show()

## Every parcel becomes a row of numbers

![features](../../anim/en/06_features.svg)


In [ ]:
# Step 4 — Per-parcel features: mean and st.dev of the 13 bands
flat_seg = seg.ravel()
counts = np.bincount(flat_seg, minlength=n_seg + 1).astype(float)
counts[counts == 0] = 1

features = np.zeros((n_seg + 1, len(band_names) * 2), dtype=np.float32)
for b in range(len(band_names)):
    vals = img[b].ravel().astype(np.float64)
    s1 = np.bincount(flat_seg, weights=vals, minlength=n_seg + 1)
    s2 = np.bincount(flat_seg, weights=vals * vals, minlength=n_seg + 1)
    mean = s1 / counts
    var = np.maximum(s2 / counts - mean**2, 0)
    features[:, 2*b], features[:, 2*b+1] = mean, np.sqrt(var)

feature_names = [f"{n}_{s}" for n in band_names for s in ("mean", "std")]
print(f"Feature table: {features.shape[0]-1} parcels x {features.shape[1]} features")

## Field labels: the ground truth (with a purity filter)

The label raster comes from **1,645 real field points** collected in the
Yaqui Valley: wheat, corn, chickpea and more.

![labels and purity](../../anim/en/07_labels_purity.svg)


In [ ]:
# Step 5 — Attach labels to parcels (majority + purity)
with rasterio.open(LABELS) as src:
    lab = src.read(1)
class_names = {int(k): v for k, v in json.load(open(NAMES)).items()}

parcel_label = np.zeros(n_seg + 1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    values = lab[(seg == sid) & (lab > 0)]
    uniq = np.unique(values)
    if len(uniq) == 1:                      # pure parcel -> usable for training
        parcel_label[sid] = uniq[0]

train_ids = np.flatnonzero(parcel_label)
print(f"Pure labeled parcels: {len(train_ids)} of {n_seg}")
for cid, cname in class_names.items():
    print(f"  {cname:12s}: {(parcel_label[train_ids] == cid).sum():3d} parcels")

## Train the classifier

![training](../../anim/en/08_training.svg)


In [ ]:
# Step 6 — Random Forest (in the browser; the full pipeline uses TPOT/AutoML)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = features[train_ids]
y = parcel_label[train_ids]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
present = sorted(np.unique(y_te))
print(classification_report(
    y_te, model.predict(X_te),
    labels=present, target_names=[class_names[c] for c in present]))

## The crop map

![crop map](../../anim/en/09_crop_map.svg)


In [ ]:
# Step 7 — Classify EVERY parcel and paint the map
pred = np.zeros(n_seg + 1, dtype=int)
pred[1:] = model.predict(features[1:])
crop_map = pred[seg]

palette = {1: "#c2703d", 2: "#65a30d", 3: "#a8a29e",
           4: "#86efac", 5: "#7c3aed", 6: "#eab308"}
rgb_map = np.zeros((*crop_map.shape, 3))
for cid, hx in palette.items():
    rgb_map[crop_map == cid] = [int(hx[i:i+2], 16)/255 for i in (1, 3, 5)]

import matplotlib.patches as mpatches
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 6))
ax1.imshow(rgb); ax1.set_title("Geomedian (true color)"); ax1.axis("off")
ax2.imshow(rgb_map); ax2.set_title("Predicted crop map"); ax2.axis("off")
ax2.legend(handles=[mpatches.Patch(color=palette[c], label=class_names[c])
                    for c in sorted(class_names)],
           loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()
print("Congratulations — you classified real crops in your browser. 🌾")

## From this workshop to production

![full pipeline](../../anim/en/10_full_pipeline.svg)

Everything you just did, **[geocrop_analysis_mx](https://github.com/abxda/geocrop_analysis_mx)**
does at scale:

| Here (browser) | Full pipeline |
|---|---|
| 1 tile, 1 month | whole regions, many months + Sentinel-1 radar |
| bundled data | live download from STAC catalogs (NASA / Planetary Computer / Earth Search) — **or optionally Google Earth Engine** |
| Random Forest | TPOT (AutoML) |
| per-band mean/std | full zonal statistics + your own rasters (DEM, climate…) as extra features |

Install it with plain `pip install -r requirements.txt` (Windows, Linux,
macOS — no conda, no compilers) and follow its step-by-step tutorial.


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Crop classification, the whole idea](https://abxda.github.io/rs-learning-audio/?id=crop-classification)
- [Crop type mapping in practice](https://abxda.github.io/rs-learning-audio/?id=crop-type-mapping)
- [Land cover, the broader family](https://abxda.github.io/rs-learning-audio/?id=land-cover)



---

[← Previous · Module 7 — Machine learning from zero](07_machine_learning_from_zero.ipynb) · [Next → · Module 9 — From your browser to production](09_from_browser_to_production.ipynb)
